<a href="https://colab.research.google.com/github/shivanilokh/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivanilokh/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* **One Row Meaning**: One unique row represents the daily organic search performance metrics (impressions, clicks, CTR) for a unique pseudonymized content item (URL) belonging to a specific client on a single calendar date.
* **Target Window**: We are using the mid-panel month of March 2026 (`month=2026-03`) to build our features and establish the baseline training dataset.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Primary Tables**: `fact_content_daily_performance`, joined with `dim_content` and `dim_clients`.
* **Target Label**: A binary proxy for high content performance — whether a page achieved a Click-Through Rate (CTR) above the median for its client slot on that day, with non-zero impressions.
* **Deliberately Excluded**: We explicitly drop all rows where `impressions = 0` because CTR becomes mathematically undefined, and we exclude the first 3 days of newly created content to avoid indexing volatility noise.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

import duckdb

# 1. Hugging Face se connect karne ke liye secret token setup
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN 'hf_your_read_token')")

# Base Path define kar rahe hain mid-panel month (March 2026) ke liye
rel_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

print("--- Query 1: Grain Check ---")
# Yeh check karega ki (report_date, client_hash, content_hash) milakar data unique hai ya nahi
q1 = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT CONCAT(report_date, '-', client_hash, '-', content_hash)) as unique_grain_count
FROM read_parquet('{rel_path}')
WHERE impressions > 0;
"""
print(con.sql(q1).to_df())

print("\n--- Query 2: Row Count & Date Range ---")
# Yeh check karega ki data sach mein March 1 se March 31, 2026 tak ka hai aur total kitne rows hain
q2 = f"""
SELECT
    MIN(report_date) as slice_start,
    MAX(report_date) as slice_end,
    COUNT(*) as slice_total_rows
FROM read_parquet('{rel_path}')
WHERE impressions > 0;
"""
print(con.sql(q2).to_df())

print("\n--- Query 3: Availability Check (IS TRUE) ---")
# IS TRUE check jo dikhayega hamare safety filters ke baad kitna data bacha
q3 = f"""
SELECT
    COUNT(*) as rows_entering,
    SUM(CASE WHEN (impressions > 0 AND clicks >= 0) IS TRUE THEN 1 ELSE 0 END) as rows_surviving
FROM read_parquet('{rel_path}');
"""
print(con.sql(q3).to_df())

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from getpass import getpass


token = getpass("Enter your Hugging Face token: ")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')"
)

rel_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

print("--- Query 1: Grain Check ---")
q1 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(report_date, '-', client_hash_id, '-', content_hash_id)) AS unique_grain_count
FROM read_parquet('{rel_path}')
WHERE gsc_impressions > 0;
"""
print(con.sql(q1).to_df())

print("\n--- Query 2: Row Count & Date Range ---")
q2 = f"""
SELECT
    MIN(report_date) AS slice_start,
    MAX(report_date) AS slice_end,
    COUNT(*) AS slice_total_rows
FROM read_parquet('{rel_path}')
WHERE gsc_impressions > 0;
"""
print(con.sql(q2).to_df())

print("\n--- Query 3: Availability Check ---")
q3 = f"""
SELECT
    COUNT(*) AS rows_entering,
    SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END) AS rows_surviving
FROM read_parquet('{rel_path}');
"""
print(con.sql(q3).to_df())

KeyboardInterrupt: Interrupted by user

In [ ]:
import duckdb
print(duckdb.__version__)

1.3.2


In [ ]:
import requests
from getpass import getpass

token = getpass("HF Token: ")

headers = {
    "Authorization": f"Bearer {token}"
}

url = "https://huggingface.co/api/datasets/FlyRank/internship-warehouse"

r = requests.get(url, headers=headers)

print("Status Code:", r.status_code)
print(r.text[:300])

HF Token: ··········
Status Code: 200
{"_id":"6a4cc0ac90ce9cc602189d11","id":"FlyRank/internship-warehouse","author":"FlyRank","sha":"50cbf7c3909d07be4d1b5906b4d09e882e5acbf2","lastModified":"2026-07-07T10:02:21.000Z","private":false,"gated":"auto","disabled":false,"tags":["language:en","license:other","size_categories:10M<n<100M","moda


In [ ]:
import duckdb
import os
from getpass import getpass

token = getpass("HF Token: ")

os.environ["HF_TOKEN"] = token

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

rel_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

print(
    con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel_path}')").fetchall()
)




HF Token: ··········


HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2026-03/data_0.parquet' (HTTP 401)

In [ ]:
import duckdb
from huggingface_hub import login

login()

con = duckdb.connect()

con.execute("""
CREATE SECRET hf_token (
    TYPE huggingface,
    PROVIDER credential_chain
);
""")

print(
    con.sql("""
    SELECT COUNT(*)
    FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    """).fetchall()
)

HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2026-03/data_0.parquet' (HTTP 401)

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Query 1: Grain Check**
* The total_rows is 3,611,061 and the unique_grain_count is also 3,611,061. Because both numbers match exactly, it proves that the dataset is perfectly unique at the grain level of report_date, client_hash_id, and content_hash_id.

**Query 2: Row Count & Date Range**
* The dataset covers the entire month of March perfectly, starting from 2026-03-01 to 2026-03-31, with a total of 3,611,061 active rows where gsc_impressions > 0.

**Query 3: Availability Check**
* Out of 9,841,378 initial rows entering the pipeline, 3,611,061 rows survived the safety filter (gsc_impressions > 0). This indicates that roughly 36.7% of the total recorded data has active visibility and impressions for this month.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.